In [74]:
# Désinstaller l'ancienne version conflictuelle
!pip uninstall -y evidently

# Installer la version compatible Python 3.12
!pip install -q "evidently==0.6.7" pandas numpy scikit-learn

print("✅ Done — Runtime > Restart session, puis passe à la cellule 2")
import os, sys, warnings, importlib, json
import pandas as pd
import numpy as np
from datetime import datetime

warnings.filterwarnings('ignore')
importlib.invalidate_caches()

try:
    from evidently import ColumnMapping
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset, DataQualityPreset
    from evidently.metrics import DatasetDriftMetric, DatasetMissingValuesMetric
    print("✅ Evidently 0.6.x importé avec succès")
except ImportError as e:
    print(f"❌ {e}")
    print("→ Rerun Cellule 1 puis Runtime > Restart session")

THRESHOLDS = {
    'data_drift_warn':       0.20,
    'data_drift_critical':   0.40,
    'prediction_drift_warn': 0.15,
    'prediction_drift_crit': 0.30,
    'missing_values_warn':   0.05,
    'missing_values_crit':   0.15,
    'model_f1_drop_warn':    0.05,
    'model_f1_drop_crit':    0.10,
}
print("📊 Seuils chargés :", THRESHOLDS)
FILES = {
    'reference':   'DataCoSupplyChain_clean.csv',
    'supply_raw':  'DataCoSupplyChainDataset.csv',
    'access_logs': 'tokenized_access_logs.csv',
    'description': 'DescriptionDataCoSupplyChain.csv',
}

import chardet

def load_csv_smart(filename):
    """Charge un CSV en détectant automatiquement encoding et séparateur."""
    # Détecter encoding
    with open(filename, 'rb') as f:
        encoding = chardet.detect(f.read(10000))['encoding'] or 'utf-8'

    # Essayer différents séparateurs
    for sep in [',', ';', '\t']:
        try:
            df = pd.read_csv(filename, encoding=encoding,
                           sep=sep, low_memory=False)
            df.dropna(axis=1, how='all', inplace=True)
            # Si une seule colonne → mauvais séparateur
            if df.shape[1] > 3:
                print(f"   sep='{sep}' encoding='{encoding}' → {df.shape[0]:,} lignes × {df.shape[1]} cols ✅")
                return df
        except:
            continue
    return None

data = {}

print("📂 Chargement des fichiers...")
for role, filename in [
    ('reference',   'DataCoSupplyChain_clean.csv'),
    ('supply_raw',  'DataCoSupplyChainDataset.csv'),
    ('access_logs', 'tokenized_access_logs.csv'),
]:
    print(f"\n🔍 [{role}] {filename}")
    df = load_csv_smart(filename)
    if df is not None:
        data[role] = df
        print(f"✅ Chargé")
    else:
        print(f"❌ Échec chargement")

df_ref = data.get('reference')
df_cur = data.get('supply_raw')
print(f"\n📋 Fichiers chargés : {list(data.keys())}")
print(f"📐 Colonnes reference  : {list(df_ref.columns[:5])}")
print(f"📐 Colonnes supply_raw : {list(df_cur.columns[:5])}")
def normalize_cols(df):
    """Normalise les noms de colonnes : lowercase + remplace espaces par _ + supprime caractères spéciaux."""
    df.columns = (df.columns
                  .str.lower()
                  .str.strip()
                  .str.replace(' ', '_', regex=False)
                  .str.replace('(', '', regex=False)
                  .str.replace(')', '', regex=False)
                  .str.replace('/', '_', regex=False)
                  .str.replace('-', '_', regex=False)
                 )
    return df

def normalize_cols(df):
    """Normalise les noms de colonnes : lowercase + remplace espaces par _ + supprime caractères spéciaux."""
    df.columns = (df.columns
                  .str.lower()
                  .str.strip()
                  .str.replace(' ', '_', regex=False)
                  .str.replace('(', '', regex=False)
                  .str.replace(')', '', regex=False)
                  .str.replace('/', '_', regex=False)
                  .str.replace('-', '_', regex=False)
                 )
    return df

# Normaliser tous les dataframes
for role in data:
    data[role] = normalize_cols(data[role])

df_ref = data.get('reference')
df_cur = data.get('supply_raw')

# Vérifier colonnes communes après normalisation
common = list(set(df_ref.columns) & set(df_cur.columns))
print(f"✅ Colonnes communes après normalisation : {len(common)}")
print(f"   Exemples : {common[:5]}")

Found existing installation: evidently 0.6.7
Uninstalling evidently-0.6.7:
  Successfully uninstalled evidently-0.6.7
✅ Done — Runtime > Restart session, puis passe à la cellule 2
✅ Evidently 0.6.x importé avec succès
📊 Seuils chargés : {'data_drift_warn': 0.2, 'data_drift_critical': 0.4, 'prediction_drift_warn': 0.15, 'prediction_drift_crit': 0.3, 'missing_values_warn': 0.05, 'missing_values_crit': 0.15, 'model_f1_drop_warn': 0.05, 'model_f1_drop_crit': 0.1}
📂 Chargement des fichiers...

🔍 [reference] DataCoSupplyChain_clean.csv
   sep=';' encoding='UTF-8-SIG' → 9,979 lignes × 56 cols ✅
✅ Chargé

🔍 [supply_raw] DataCoSupplyChainDataset.csv
   sep=';' encoding='UTF-8-SIG' → 7,992 lignes × 52 cols ✅
✅ Chargé

🔍 [access_logs] tokenized_access_logs.csv
   sep=';' encoding='UTF-8-SIG' → 10,274 lignes × 8 cols ✅
✅ Chargé

📋 Fichiers chargés : ['reference', 'supply_raw', 'access_logs']
📐 Colonnes reference  : ['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_orde

In [75]:
SAMPLE = 1000

def get_valid_cols(df_a, df_b, sample=1000):
    common = list(set(df_a.columns) & set(df_b.columns))

    if len(common) == 0:
        print("   ⚠️ Aucune colonne commune trouvée")
        return [], pd.DataFrame(), pd.DataFrame()

    a_s = df_a[common].head(sample).copy()
    b_s = df_b[common].head(sample).copy()

    for c in common:
        a_s[c] = pd.to_numeric(a_s[c], errors='coerce')
        b_s[c] = pd.to_numeric(b_s[c], errors='coerce')

    a_s = a_s.fillna(0)
    b_s = b_s.fillna(0)

    valid = [
        c for c in common
        if a_s[c].dtype in ['float64','int64']
        and a_s[c].std() > 0
        and b_s[c].std() > 0
        and a_s[c].notna().sum() > 10
        and b_s[c].notna().sum() > 10
    ]

    print(f"   Colonnes communes : {len(common)} | valides : {len(valid)}")
    return valid, a_s[valid], b_s[valid]

all_drift_results = {}

# ── 1/3 : clean vs DataCoSupplyChainDataset ───────────────────────────────────
print("="*55)
print("🔍 [1/3] Clean vs DataCoSupplyChainDataset")
print("="*55)
valid_cols_1, ref1, cur1 = get_valid_cols(data['reference'], data['supply_raw'])

if len(valid_cols_1) >= 2:
    r1 = Report(metrics=[DatasetDriftMetric(), DatasetMissingValuesMetric(), DataDriftPreset()])
    r1.run(reference_data=ref1, current_data=cur1)
    r1.show(mode='inline')
    all_drift_results['supply_raw'] = r1.as_dict()
    print("✅ Terminé\n")
else:
    print("⚠️ 0 colonnes valides — diagnostic:")
    common_debug = list(set(data['reference'].columns) & set(data['supply_raw'].columns))
    print(f"   Colonnes communes : {common_debug[:5]}")
    print(f"   Types reference  : {data['reference'][common_debug[:3]].dtypes.to_dict()}")
    print(f"   Types supply_raw : {data['supply_raw'][common_debug[:3]].dtypes.to_dict()}")
    all_drift_results['supply_raw'] = {}
    ref1 = data['reference'].head(SAMPLE)
    cur1 = data['supply_raw'].head(SAMPLE)

# ── 2/3 : clean vs tokenized_access_logs ──────────────────────────────────────
print("="*55)
print("🔍 [2/3] Clean vs tokenized_access_logs")
print("="*55)

# Vérifier si access_logs est chargé
if 'access_logs' in data:
    valid_cols_2, ref2, cur2 = get_valid_cols(data['reference'], data['access_logs'])
    if len(valid_cols_2) >= 2:
        r2 = Report(metrics=[DatasetDriftMetric(), DatasetMissingValuesMetric(), DataDriftPreset()])
        r2.run(reference_data=ref2, current_data=cur2)
        r2.show(mode='inline')
        all_drift_results['access_logs'] = r2.as_dict()
        print("✅ Terminé\n")
    else:
        print("⚠️ Pas assez de colonnes numériques communes — skipped\n")
        ref2, cur2 = ref1, cur1
        valid_cols_2 = []
else:
    print("⚠️ access_logs non chargé — skipped\n")
    ref2, cur2 = ref1, cur1
    valid_cols_2 = []

# ── 3/3 : description skipped ─────────────────────────────────────────────────
print("="*55)
print("🔍 [3/3] DescriptionDataCoSupplyChain → fichier texte, skipped")
print("="*55)

# Variables pour cellules 6 & 7
drift_results = all_drift_results.get('supply_raw', {})
ref_sample    = ref1
cur_sample    = cur1
print("\n📌 Référence principale pour alertes = clean vs supply_raw")

🔍 [1/3] Clean vs DataCoSupplyChainDataset
   Colonnes communes : 47 | valides : 27
✅ Terminé

🔍 [2/3] Clean vs tokenized_access_logs
   ⚠️ Aucune colonne commune trouvée
⚠️ Pas assez de colonnes numériques communes — skipped

🔍 [3/3] DescriptionDataCoSupplyChain → fichier texte, skipped

📌 Référence principale pour alertes = clean vs supply_raw


In [76]:
print("🔍 Qualité : Clean vs DataCoSupplyChainDataset")
q1 = Report(metrics=[DataQualityPreset()])
q1.run(reference_data=ref1, current_data=cur1)
q1.show(mode='inline')
print("✅ [1/2] Terminé\n")

if len(valid_cols_2) >= 2:
    print("🔍 Qualité : Clean vs tokenized_access_logs")
    q2 = Report(metrics=[DataQualityPreset()])
    q2.run(reference_data=ref2, current_data=cur2)
    q2.show(mode='inline')
    print("✅ [2/2] Terminé\n")

quality_report = q1  # utilisé dans cellule 7 pour l'export HTML
print("✅ Tous les rapports qualité terminés")

🔍 Qualité : Clean vs DataCoSupplyChainDataset
✅ [1/2] Terminé

✅ Tous les rapports qualité terminés


In [77]:
def safe_get(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict): return default
        d = d.get(k, default)
        if d is default: return default
    return d

def alert_level(value, warn, crit):
    if value >= crit:  return 'CRITICAL'
    if value >= warn:  return 'WARNING'
    return 'HEALTHY'

drift_share   = 0.0
missing_share = 0.0

for m in drift_results.get('metrics', []):
    metric_name = str(m.get('metric', ''))
    if 'DatasetDriftMetric' in metric_name:
        drift_share = safe_get(m, 'result', 'drift_share', default=0.0)
    if 'DatasetMissingValuesMetric' in metric_name:
        missing_share = safe_get(m, 'result', 'current', 'share_of_missing_values', default=0.0)

# ── Forcer numérique sur ref_sample et cur_sample ──
ref_numeric = ref_sample.apply(pd.to_numeric, errors='coerce').fillna(0)
cur_numeric = cur_sample.apply(pd.to_numeric, errors='coerce').fillna(0)

# ── Prediction drift ──
first_col = ref_numeric.columns[0]
pred_drift = abs(ref_numeric[first_col].mean() - cur_numeric[first_col].mean())
pred_drift = min(pred_drift / (ref_numeric[first_col].std() * 5 + 1e-9), 1.0)

# ── Model degradation ──
target_col = next(
    (c for c in ref_numeric.columns if 'risk' in c.lower() or 'target' in c.lower()),
    None
)
if target_col:
    model_deg = abs(ref_numeric[target_col].mean() - cur_numeric[target_col].mean())
else:
    model_deg = round(drift_share * 0.8, 4)

now = datetime.now()

record = {
    'Date':                     now.strftime('%d/%m/%Y'),
    'Time':                     now.strftime('%H:%M:%S'),
    'Timestamp':                now.isoformat(),
    'DataDriftScore_Pct':       round(drift_share * 100, 2),
    'DataDriftStatus':          alert_level(drift_share,   THRESHOLDS['data_drift_warn'],      THRESHOLDS['data_drift_critical']),
    'PredictionDriftScore_Pct': round(pred_drift * 100, 2),
    'PredictionDriftStatus':    alert_level(pred_drift,    THRESHOLDS['prediction_drift_warn'], THRESHOLDS['prediction_drift_crit']),
    'MissingValues_Pct':        round(missing_share * 100, 2),
    'MissingValuesStatus':      alert_level(missing_share, THRESHOLDS['missing_values_warn'],   THRESHOLDS['missing_values_crit']),
    'ModelDegradation_Pct':     round(model_deg * 100, 2),
    'ModelDegradationStatus':   alert_level(model_deg,     THRESHOLDS['model_f1_drop_warn'],    THRESHOLDS['model_f1_drop_crit']),
    'RetrainingRequired':       'YES' if any([
        drift_share   >= THRESHOLDS['data_drift_critical'],
        pred_drift    >= THRESHOLDS['prediction_drift_crit'],
        missing_share >= THRESHOLDS['missing_values_crit'],
        model_deg     >= THRESHOLDS['model_f1_drop_crit'],
    ]) else 'NO',
    'N_ReferenceRows':          len(df_ref),
    'N_CurrentRows':            len(df_cur),
    'Threshold_DataDrift_Warn':     THRESHOLDS['data_drift_warn'] * 100,
    'Threshold_DataDrift_Critical': THRESHOLDS['data_drift_critical'] * 100,
    'Threshold_PredDrift_Warn':     THRESHOLDS['prediction_drift_warn'] * 100,
    'Threshold_PredDrift_Critical': THRESHOLDS['prediction_drift_crit'] * 100,
    'Threshold_ModelDeg_Warn':      THRESHOLDS['model_f1_drop_warn'] * 100,
    'Threshold_ModelDeg_Critical':  THRESHOLDS['model_f1_drop_crit'] * 100,
}

print("═"*55)
print("       📊 MONITORING DASHBOARD SUMMARY")
print("═"*55)
print(f"  🕐 Timestamp         : {record['Timestamp']}")
print(f"  📈 Data Drift        : {record['DataDriftScore_Pct']}%  → {record['DataDriftStatus']}")
print(f"  🎯 Prediction Drift  : {record['PredictionDriftScore_Pct']}%  → {record['PredictionDriftStatus']}")
print(f"  ❓ Missing Values    : {record['MissingValues_Pct']}%  → {record['MissingValuesStatus']}")
print(f"  📉 Model Degradation : {record['ModelDegradation_Pct']}%  → {record['ModelDegradationStatus']}")
print("─"*55)
print(f"  🚀 RETRAINING REQUIRED : {record['RetrainingRequired']}")
print("═"*55)

═══════════════════════════════════════════════════════
       📊 MONITORING DASHBOARD SUMMARY
═══════════════════════════════════════════════════════
  🕐 Timestamp         : 2026-03-14T15:38:36.980084
  📈 Data Drift        : 50.0%  → CRITICAL
  🎯 Prediction Drift  : 0.67%  → HEALTHY
  ❓ Missing Values    : 0.0%  → HEALTHY
  📉 Model Degradation : 2.4%  → HEALTHY
───────────────────────────────────────────────────────
  🚀 RETRAINING REQUIRED : YES
═══════════════════════════════════════════════════════


In [78]:
print("Colonnes supply_raw:")
print(list(data['supply_raw'].columns))
print()
print("Shape:", data['supply_raw'].shape)
print()
print("Colonnes reference:")
print(list(data['reference'].columns))

Colonnes supply_raw:
['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date_dateorders', 'shi

In [80]:
from google.colab import files

POWERBI_CSV = 'monitoring_results_powerbi.csv'

pbi_df = pd.DataFrame([record])

if os.path.exists(POWERBI_CSV):
    existing = pd.read_csv(POWERBI_CSV)
    pbi_df = pd.concat([existing, pbi_df], ignore_index=True)
    print(f"📎 Ligne ajoutée — total: {len(pbi_df)} enregistrements")
else:
    print("📄 Nouveau fichier créé")

pbi_df.to_csv(POWERBI_CSV, index=False)

# Sauvegarder HTML seulement si les rapports existent
if 'r1' in dir():
    r1.save_html('data_drift_report.html')
    print("✅ data_drift_report.html")
else:
    print("⚠️ data_drift_report.html — skipped (rapport non généré)")

if 'q1' in dir():
    q1.save_html('data_quality_report.html')
    print("✅ data_quality_report.html")
else:
    print("⚠️ data_quality_report.html — skipped (rapport non généré)")

print(f"✅ {POWERBI_CSV}")
print("\n⬇️ Téléchargement...")
files.download(POWERBI_CSV)

print("\n─"*28)
print("📌 POWER BI — importer le CSV:")
print("  Home → Get Data → Text/CSV → monitoring_results_powerbi.csv")
print("  Colonnes KPI  : DataDriftScore_Pct, PredictionDriftScore_Pct,")
print("                  ModelDegradation_Pct, MissingValues_Pct")
print("  Line chart    : Timestamp (X) + *_Pct (Y)")
print("  Slicer        : RetrainingRequired")

📎 Ligne ajoutée — total: 2 enregistrements
✅ data_drift_report.html
✅ data_quality_report.html
✅ monitoring_results_powerbi.csv

⬇️ Téléchargement...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
📌 POWER BI — importer le CSV:
  Home → Get Data → Text/CSV → monitoring_results_powerbi.csv
  Colonnes KPI  : DataDriftScore_Pct, PredictionDriftScore_Pct,
                  ModelDegradation_Pct, MissingValues_Pct
  Line chart    : Timestamp (X) + *_Pct (Y)
  Slicer        : RetrainingRequired


In [81]:
import json

grafana_dashboard = {
    "title": "ML Monitoring — Supply Chain Drift",
    "uid": "supply-chain-ml",
    "schemaVersion": 36,
    "refresh": "5m",
    "tags": ["ml", "drift", "supply-chain"],
    "panels": [
        {
            "id": 1, "type": "stat",
            "title": "Data Drift Score (%)",
            "description": "% features avec distribution shift vs reference",
            "fieldConfig": {"defaults": {
                "thresholds": {"steps": [
                    {"color": "green",  "value": 0},
                    {"color": "yellow", "value": 20},
                    {"color": "red",    "value": 40}
                ]},
                "unit": "percent", "min": 0, "max": 100
            }},
            "targets": [{"expr": "DataDriftScore_Pct"}],
            "gridPos": {"h": 4, "w": 6, "x": 0, "y": 0}
        },
        {
            "id": 2, "type": "stat",
            "title": "Prediction Drift Score (%)",
            "fieldConfig": {"defaults": {
                "thresholds": {"steps": [
                    {"color": "green",  "value": 0},
                    {"color": "yellow", "value": 15},
                    {"color": "red",    "value": 30}
                ]},
                "unit": "percent", "min": 0, "max": 100
            }},
            "targets": [{"expr": "PredictionDriftScore_Pct"}],
            "gridPos": {"h": 4, "w": 6, "x": 6, "y": 0}
        },
        {
            "id": 3, "type": "stat",
            "title": "Model Degradation (%)",
            "fieldConfig": {"defaults": {
                "thresholds": {"steps": [
                    {"color": "green",  "value": 0},
                    {"color": "yellow", "value": 5},
                    {"color": "red",    "value": 10}
                ]},
                "unit": "percent", "min": 0, "max": 100
            }},
            "targets": [{"expr": "ModelDegradation_Pct"}],
            "gridPos": {"h": 4, "w": 6, "x": 12, "y": 0}
        },
        {
            "id": 4, "type": "stat",
            "title": "Missing Values (%)",
            "fieldConfig": {"defaults": {
                "thresholds": {"steps": [
                    {"color": "green",  "value": 0},
                    {"color": "yellow", "value": 5},
                    {"color": "red",    "value": 15}
                ]},
                "unit": "percent", "min": 0, "max": 100
            }},
            "targets": [{"expr": "MissingValues_Pct"}],
            "gridPos": {"h": 4, "w": 6, "x": 18, "y": 0}
        },
        {
            "id": 5, "type": "timeseries",
            "title": "Drift Over Time",
            "targets": [
                {"expr": "DataDriftScore_Pct",       "legendFormat": "Data Drift"},
                {"expr": "PredictionDriftScore_Pct", "legendFormat": "Prediction Drift"},
                {"expr": "ModelDegradation_Pct",     "legendFormat": "Model Degradation"}
            ],
            "gridPos": {"h": 8, "w": 24, "x": 0, "y": 4}
        },
        {
            "id": 6, "type": "table",
            "title": "Monitoring Log Complet",
            "targets": [{"expr": "monitoring_results_powerbi"}],
            "gridPos": {"h": 8, "w": 24, "x": 0, "y": 12}
        },
        {
            "id": 7, "type": "text",
            "title": "Retraining Status",
            "options": {"content": "### Retraining Required\n${RetrainingRequired}"},
            "gridPos": {"h": 4, "w": 24, "x": 0, "y": 20}
        }
    ],
    "templating": {"list": [
        {
            "name": "RetrainingRequired",
            "type": "query",
            "query": "RetrainingRequired"
        }
    ]},
    "annotations": {"list": [
        {
            "name": "Retraining Triggered",
            "expr": "RetrainingRequired == 'YES'",
            "titleFormat": "🚨 RETRAIN",
            "iconColor": "red"
        }
    ]}
}

with open('grafana_dashboard.json', 'w') as f:
    json.dump(grafana_dashboard, f, indent=2)

print("✅ grafana_dashboard.json exporté")
print()
print("📌 COMMENT IMPORTER DANS GRAFANA:")
print("   1. Ouvre Grafana → menu gauche → Dashboards → Import")
print("   2. Upload grafana_dashboard.json")
print("   3. Datasource → sélectionne CSV datasource → pointe vers monitoring_results_powerbi.csv")
print("   4. Save → dashboard live avec refresh toutes les 5 min")

from google.colab import files
files.download('grafana_dashboard.json')

✅ grafana_dashboard.json exporté

📌 COMMENT IMPORTER DANS GRAFANA:
   1. Ouvre Grafana → menu gauche → Dashboards → Import
   2. Upload grafana_dashboard.json
   3. Datasource → sélectionne CSV datasource → pointe vers monitoring_results_powerbi.csv
   4. Save → dashboard live avec refresh toutes les 5 min


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [82]:
# ─── AWS S3 + GCP GCS — Scaling Architecture ──────────────────────────────────
# Ce code montre l'intégration cloud pour la production
# Remplace les credentials par les vrais pour activer l'upload réel

# ══════════════════════════════════════════════════════
#  AWS S3 UPLOAD
# ══════════════════════════════════════════════════════
AWS_ACCESS_KEY    = ''       # ton AWS Access Key ID
AWS_SECRET_KEY    = ''       # ton AWS Secret Access Key
AWS_BUCKET_NAME   = 'supply-chain-ml-monitoring'
AWS_REGION        = 'us-east-1'

def upload_to_s3(local_file, bucket, region, access_key, secret_key):
    if not all([access_key, secret_key]):
        print("⚠️  AWS — credentials non renseignés")
        print("   → Remplis AWS_ACCESS_KEY et AWS_SECRET_KEY pour activer")
        print("   → Architecture: Colab → S3 → CloudWatch → Lambda → SageMaker Retrain")
        return False
    try:
        import boto3
        s3 = boto3.client(
            's3',
            region_name=region,
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key
        )
        s3.upload_file(local_file, bucket, f'monitoring/{local_file}')
        print(f"✅ AWS S3 — uploadé → s3://{bucket}/monitoring/{local_file}")
        return True
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'boto3'])
        print("boto3 installé — relance cette cellule")
    except Exception as e:
        print(f"❌ AWS S3 erreur: {e}")
    return False

# ══════════════════════════════════════════════════════
#  GCP GCS UPLOAD
# ══════════════════════════════════════════════════════
GCP_BUCKET_NAME       = 'supply-chain-ml-monitoring'
GCP_CREDENTIALS_FILE  = ''   # chemin vers ton fichier JSON credentials GCP

def upload_to_gcs(local_file, bucket, credentials_file):
    if not credentials_file:
        print("⚠️  GCP — credentials non renseignés")
        print("   → Remplis GCP_CREDENTIALS_FILE pour activer")
        print("   → Architecture: Colab → GCS → Cloud Monitoring → Pub/Sub → Vertex AI Retrain")
        return False
    try:
        from google.cloud import storage
        client = storage.Client.from_service_account_json(credentials_file)
        bucket_obj = client.bucket(bucket)
        blob = bucket_obj.blob(f'monitoring/{local_file}')
        blob.upload_from_filename(local_file)
        print(f"✅ GCP GCS — uploadé → gs://{bucket}/monitoring/{local_file}")
        return True
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'google-cloud-storage'])
        print("google-cloud-storage installé — relance cette cellule")
    except Exception as e:
        print(f"❌ GCP GCS erreur: {e}")
    return False

# ── Lancer les uploads ────────────────────────────────
print("☁️  CLOUD UPLOAD — monitoring_results_powerbi.csv")
print("─"*55)
upload_to_s3(POWERBI_CSV, AWS_BUCKET_NAME, AWS_REGION, AWS_ACCESS_KEY, AWS_SECRET_KEY)
upload_to_gcs(POWERBI_CSV, GCP_BUCKET_NAME, GCP_CREDENTIALS_FILE)

print()
print("─"*55)
print("📐 SCALING ARCHITECTURE — 10x Data Growth")
print("─"*55)
print("""
  PIPELINE COMPLET:

  [Google Colab]
       │ Evidently génère métriques drift
       ↓
  [AWS S3 / GCP GCS]
       │ stockage monitoring_results_powerbi.csv
       ↓
  [CloudWatch / Cloud Monitoring]
       │ détecte seuil critique (drift > 40%)
       ↓
  [Lambda / Cloud Function]
       │ déclenche retraining automatique
       ↓
  [SageMaker / Vertex AI]
       │ retraining sur nouvelles données
       ↓
  [Model Registry]
       │ nouveau modèle validé
       ↓
  [Grafana Dashboard]
       │ visualise drift + alertes en temps réel
       ↓
  [Power BI]
       └ reporting business pour les stakeholders

  SCALING 10x:
  • Données: S3/GCS partitionné par date (Parquet)
  • Compute: Auto-scaling EC2 / Cloud Run
  • Training: Spot instances AWS / Preemptible GCP (70% économie)
  • Monitoring: Serverless Lambda/Cloud Functions
""")

☁️  CLOUD UPLOAD — monitoring_results_powerbi.csv
───────────────────────────────────────────────────────
⚠️  AWS — credentials non renseignés
   → Remplis AWS_ACCESS_KEY et AWS_SECRET_KEY pour activer
   → Architecture: Colab → S3 → CloudWatch → Lambda → SageMaker Retrain
⚠️  GCP — credentials non renseignés
   → Remplis GCP_CREDENTIALS_FILE pour activer
   → Architecture: Colab → GCS → Cloud Monitoring → Pub/Sub → Vertex AI Retrain

───────────────────────────────────────────────────────
📐 SCALING ARCHITECTURE — 10x Data Growth
───────────────────────────────────────────────────────

  PIPELINE COMPLET:

  [Google Colab]
       │ Evidently génère métriques drift
       ↓
  [AWS S3 / GCP GCS]
       │ stockage monitoring_results_powerbi.csv
       ↓
  [CloudWatch / Cloud Monitoring]
       │ détecte seuil critique (drift > 40%)
       ↓
  [Lambda / Cloud Function]
       │ déclenche retraining automatique
       ↓
  [SageMaker / Vertex AI]
       │ retraining sur nouvelles données
   

In [83]:
print("""
╔══════════════════════════════════════════════════════════════╗
║         SCALING ARCHITECTURE — 10x DATA GROWTH              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  CURRENT (1x)          →      SCALED (10x)                   ║
║  ─────────────────────────────────────────────────────────  ║
║  CSV batch             →      Kafka streaming                ║
║  Single node           →      Spark distributed (EMR)        ║
║  Manual drift check    →      Evidently auto-scheduled       ║
║  Local storage         →      S3 / GCS data lake             ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  DATA LAYER                                                  ║
║  ─────────────────────────────────────────────────────────  ║
║  AWS : S3 → Glue ETL → Redshift                             ║
║  GCP : GCS → Dataflow → BigQuery                            ║
║  Format : Parquet + partition par date                       ║
╠══════════════════════════════════════════════════════════════╣
║  MONITORING LAYER                                            ║
║  ─────────────────────────────────────────────────────────  ║
║  Evidently  →  génère métriques JSON toutes les 5 min       ║
║  CloudWatch →  détecte seuil critique                       ║
║  Grafana    →  dashboard temps réel                         ║
║  Power BI   →  reporting business                           ║
╠══════════════════════════════════════════════════════════════╣
║  RETRAINING LAYER                                            ║
║  ─────────────────────────────────────────────────────────  ║
║  Trigger   : drift > 40%  OU  dégradation > 10%             ║
║  AWS       : Lambda → SageMaker Pipeline → Model Registry   ║
║  GCP       : Pub/Sub → Cloud Function → Vertex AI           ║
║  Validation: shadow testing nouveau vs ancien modèle        ║
║  Deploy    : Blue/Green 5% → 50% → 100% traffic             ║
║  Rollback  : auto si dégradation > 2% vs champion           ║
╠══════════════════════════════════════════════════════════════╣
║  COMPUTE SCALING                                             ║
║  ─────────────────────────────────────────────────────────  ║
║  Inference  : Auto-scaling EC2 / Cloud Run (CPU cible 60%)  ║
║  Training   : Spot instances (économie 70%)                 ║
║  Monitoring : Serverless Lambda / Cloud Functions           ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║         SCALING ARCHITECTURE — 10x DATA GROWTH              ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  CURRENT (1x)          →      SCALED (10x)                   ║
║  ─────────────────────────────────────────────────────────  ║
║  CSV batch             →      Kafka streaming                ║
║  Single node           →      Spark distributed (EMR)        ║
║  Manual drift check    →      Evidently auto-scheduled       ║
║  Local storage         →      S3 / GCS data lake             ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  DATA LAYER                                                  ║
║  ─────────────────────────────────────────────────────────  ║
║  AWS : S3 → Glue ETL → Redshift                             ║
║  GCP : GCS → Dataflow → Bi